# Heart Disease Random Forest + RandomizedSearchCV

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report,accuracy_score,precision_score,recall_score,f1_score,
balanced_accuracy_score,matthews_corrcoef,roc_auc_score,average_precision_score,
ConfusionMatrixDisplay,RocCurveDisplay,PrecisionRecallDisplay)


In [ ]:
url='https://raw.githubusercontent.com/btenneson/public_projects/refs/heads/main/predator/heart_disease_health_indicators_BRFSS2015.csv'
df=pd.read_csv(url)
X=df.iloc[:,1:]
y=df.iloc[:,0]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)

pipe=Pipeline([
('imputer',SimpleImputer(strategy='median')),
('model',RandomForestClassifier(random_state=42,n_jobs=-1))
])

param_dist={
'model__n_estimators':[200,400,600,800,1000],
'model__max_depth':[None,10,20,40],
'model__min_samples_split':[2,5,10],
'model__min_samples_leaf':[1,2,4],
'model__max_features':['sqrt','log2',None],
'model__bootstrap':[True,False],
'model__class_weight':[None,'balanced','balanced_subsample']
}

cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

search=RandomizedSearchCV(pipe,param_dist,n_iter=50,scoring='roc_auc',cv=cv,n_jobs=-1,random_state=42,refit=True)
search.fit(X_train,y_train)
best=search.best_estimator_

cv_prob=cross_val_predict(best,X_train,y_train,cv=cv,method='predict_proba',n_jobs=-1)[:,1]

ths=np.linspace(0.01,0.99,991)
best_t=0.5
best_mcc=-1
for t in ths:
    p=(cv_prob>=t).astype(int)
    m=matthews_corrcoef(y_train,p)
    if m>best_mcc:
        best_mcc=m
        best_t=t

prob=best.predict_proba(X_test)[:,1]
pred=(prob>=best_t).astype(int)

print(search.best_params_)
print(classification_report(y_test,pred))
print('Accuracy',accuracy_score(y_test,pred))
print('Precision',precision_score(y_test,pred,zero_division=0))
print('Recall',recall_score(y_test,pred,zero_division=0))
print('F1',f1_score(y_test,pred,zero_division=0))
print('Balanced Accuracy',balanced_accuracy_score(y_test,pred))
print('MCC',matthews_corrcoef(y_test,pred))
print('ROC-AUC',roc_auc_score(y_test,prob))
print('PR-AUC',average_precision_score(y_test,prob))

ConfusionMatrixDisplay.from_predictions(y_test,pred)
plt.show()
RocCurveDisplay.from_predictions(y_test,prob)
plt.show()
PrecisionRecallDisplay.from_predictions(y_test,prob)
plt.show()
